# FMA-full: from the catalogue to train, validation and test

This notebook follows the data from the published FMA-full catalogue to the datasets used in the experiments:

1. the original catalogue: label hierarchy, imbalance and artist concentration;
2. the audio audit, with examples of problematic files;
3. which tracks are retained, excluded or left unresolved, and what that changes for each genre;
4. how tracks are grouped and split;
5. the published FMA split compared with ours;
6. the primary benchmark, supplementary views and remaining limitations.

The saved tables are aggregates produced on WCSS. To reproduce them, run the audit and both splits as CPU jobs from the repository root, then open this notebook in OOD with the **Music Atlas** kernel (`docs/DEVELOPMENT.md`):

```bash
sbatch scripts/slurm/fma-audit.sbatch
AUDIT="$ATLAS_DATA_DIR/manifests/fma_audio_audit.parquet"
sbatch scripts/slurm/fma-split.sbatch --audit "$AUDIT"
sbatch scripts/slurm/fma-split.sbatch --audit "$AUDIT" --strategy rare-support
```

`VERSION` is the primary dataset and `COMPARE_VERSION` the rare-support partition of the same tracks. Both names are the short links under `$ATLAS_DATA_DIR/ready/`.

In [ ]:
import io
import json
import tarfile
from ast import literal_eval
from html import escape

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML, Audio, display
from matplotlib.ticker import PercentFormatter

from music_atlas import audit as rules
from music_atlas.loader import load_windows
from music_atlas.settings import paths, profile

VERSION = "fma-v1"
COMPARE_VERSION = "fma-rare-support-v1"

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "#f8fafc",
    "axes.labelcolor": "#24334a",
    "axes.edgecolor": "#d4deea",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titleweight": "bold",
    "axes.titlepad": 12,
    "text.color": "#17253d",
    "xtick.color": "#3b4d65",
    "ytick.color": "#3b4d65",
    "grid.color": "#dbe4ed",
    "axes.axisbelow": True,
    "figure.dpi": 110,
})

data = paths()["data"]
metadata = data / "fma/fma_metadata"
inventory = pd.read_csv(
    data / "manifests/fma_full_inventory.csv",
    usecols=["track_id", "shard", "audio_path", "status", "sha1"],
    index_col="track_id",
)
assert inventory.index.is_unique and not inventory.index.hasnans
ids = inventory.index[inventory["status"].eq("present")]
assert len(ids), "The inventory lists no present tracks."
fields = {
    ("artist", "id"): "artist_id",
    ("artist", "name"): "artist",
    ("album", "id"): "album_id",
    ("set", "split"): "split",
    ("set", "subset"): "subset",
    ("track", "title"): "title",
    ("track", "duration"): "duration",
    ("track", "genre_top"): "genre_top",
    ("track", "genres"): "genres",
    ("track", "genres_all"): "genres_all",
}
tracks = pd.read_csv(metadata / "tracks.csv", index_col=0, header=[0, 1], low_memory=False)
assert tracks.index.is_unique
missing_rows = len(ids.difference(tracks.index))
tracks = tracks[list(fields)].reindex(ids)
tracks.columns = list(fields.values())
for field in ["artist_id", "album_id"]:
    tracks[field] = tracks[field].where(tracks[field] > 0)
print(f"{len(tracks):,} tracks in the inventory, {missing_rows:,} without a metadata row.")

## 1. The original catalogue

FMA-full contains full-length MP3s. Each **track_id** links an audio file to a row in **tracks.csv**, which holds artist and album IDs, duration, genre tags and the published split and subset. **genres.csv** gives each genre's name and parent. This section describes every track in the inventory, before any audio check. The track map also reads FMA's published **features.csv**. [FMA data documentation](https://github.com/mdeff/fma#data)

### Label hierarchy

Expand a branch to see its children, then expand a child to go deeper. The first level lists every top-level **Branch** in FMA. A **Parent** is one level above a genre.

**Direct** counts tracks explicitly tagged with that genre in **genres**. **With descendants** counts tracks tagged with the genre or anything below it in the tree, using **genres_all**. A track tagged only Post-Punk counts directly under Post-Punk and toward the inclusive totals for Punk and Rock. A track with both Punk and Post-Punk tags contributes only once to the Rock total. A track can contribute to several branches. [FMA dataset paper](https://arxiv.org/html/1612.01840v3#S2.SS4)

In [ ]:
genres = pd.read_csv(metadata / "genres.csv", index_col="genre_id")
labels = tracks[["genres", "genres_all"]].fillna("[]").map(literal_eval)
direct = labels["genres"].explode().dropna().reset_index(name="genre_id").drop_duplicates()
inclusive = labels["genres_all"].explode().dropna().reset_index(name="genre_id").drop_duplicates()
assert genres.index.is_unique
assert (
    direct["genre_id"].isin(genres.index).all() and inclusive["genre_id"].isin(genres.index).all()
)
assert pd.MultiIndex.from_frame(direct).isin(pd.MultiIndex.from_frame(inclusive)).all()

genres["Direct"] = direct["genre_id"].value_counts().reindex(genres.index, fill_value=0)
genres["With descendants"] = (
    inclusive["genre_id"].value_counts().reindex(genres.index, fill_value=0)
)


roots = genres[genres["parent"].eq(0)]
children_by_parent = genres.sort_values("title").groupby("parent").groups


def genre_node(genre_id):
    row = genres.loc[genre_id]
    name = escape(row["title"])
    counts = f"{row['Direct']:,} direct · {row['With descendants']:,} with descendants"
    subgenres = children_by_parent.get(genre_id, ())
    if not len(subgenres):
        return f"{name} — {counts}"
    items = "".join(f"<li>{genre_node(child)}</li>" for child in subgenres)
    summary = f"<summary><strong>{name}</strong> — {counts}</summary>"
    return f"<details>{summary}<ul>{items}</ul></details>"


tree = "".join(f"<li>{genre_node(root)}</li>" for root in roots.sort_values("title").index)
display(HTML(f"<ul>{tree}</ul>"))

root_tags = inclusive[inclusive["genre_id"].isin(roots.index)]
roots_per_track = root_tags.groupby("track_id").size().reindex(ids, fill_value=0)


def plot_genre_counts(rows, title):
    fig, ax = plt.subplots(figsize=(8, 1.5 + 0.31 * len(rows)), layout="constrained")
    rows.sort_values("With descendants").set_index("title")[
        ["Direct", "With descendants"]
    ].plot.barh(ax=ax, color=["#5352a3", "#2a9d8f"], width=0.76, title=title)
    ax.set(xlabel="Unique tracks", ylabel="")
    ax.xaxis.set_major_formatter("{x:,.0f}")
    ax.grid(axis="x")
    ax.legend(loc="lower right", facecolor="white")
    plt.show()

### Top-level genres and imbalance

Each pair of bars compares explicit tags with the total that includes subgenres. A large gap shows that much of a branch's support comes from more specific tags. Tracks can contribute to more than one branch.

In [ ]:
plot_genre_counts(roots, "Top-level genres")

The table compares support for each top-level branch in FMA-full with FMA's own single-label subsets, which use **genre_top**. **Artists** counts distinct known artists in the branch and **Top artist** the share of its tracks from the largest one. A branch with many tracks from few artists offers less independent evidence than its size suggests.

In [ ]:
branch_tracks = root_tags.join(tracks["artist_id"], on="track_id")
branch_stats = branch_tracks.groupby("genre_id").agg(
    Tracks=("track_id", "size"), Artists=("artist_id", "nunique"),
)
largest = branch_tracks.groupby(["genre_id", "artist_id"]).size().groupby(level=0).max()
branch_stats["Top artist (%)"] = (100 * largest / branch_stats["Tracks"]).round(1)
branch_stats = branch_stats.reindex(roots.index).sort_values("Tracks", ascending=False)
branch_stats.index = pd.Index(genres.loc[branch_stats.index, "title"], name="FMA branch")
small = tracks.loc[tracks["subset"].eq("small"), "genre_top"].value_counts()
medium = tracks.loc[tracks["subset"].isin(["small", "medium"]), "genre_top"].value_counts()
balance = branch_stats.rename(columns={"Tracks": "Full tracks"})
balance["Small tracks"] = small.reindex(balance.index, fill_value=0)
balance["Medium tracks"] = medium.reindex(balance.index, fill_value=0)
display(balance)

### Which direct labels appear together?

Rows and columns show the ten most frequent **direct** labels. Each cell shows the percentage of tracks with **both** labels among the tracks with **at least one** of them. Suppose 60 tracks have tag A, 50 have tag B and 10 have both. There are 60 + 50 − 10 = 100 unique tracks with at least one tag. The **Jaccard overlap** is 10 / 100 = 10%.

Darker colours show greater overlap. The diagonal is blank because each label would overlap perfectly with itself. Counting explicit tags keeps parent–child relationships from automatically creating overlap. The heatmap measures shared annotations, and FMA's tags may be incomplete.

In [ ]:
def label_overlap(tags, genre_ids):
    flags = pd.crosstab(tags["track_id"], tags["genre_id"]).reindex(columns=genre_ids)
    flags = flags.to_numpy(dtype=np.int64)
    shared = flags.T @ flags
    totals = shared.diagonal()
    return shared / (totals + totals[:, None] - shared)


common = genres[genres["Direct"].gt(0)].nlargest(10, "Direct").index
if len(common) >= 2:
    selected = direct[direct["genre_id"].isin(common)]
    similarity = label_overlap(selected, common)
    np.fill_diagonal(similarity, np.nan)

    fig, ax = plt.subplots(figsize=(8.8, 7.4), layout="constrained")
    cmap = plt.get_cmap("YlGnBu").with_extremes(bad="#e7edf4")
    vmax = np.nanmax(similarity) or 1
    heatmap = ax.imshow(similarity, cmap=cmap, vmin=0, vmax=vmax)
    for (row, col), value in np.ndenumerate(similarity):
        if np.isfinite(value):
            ax.text(
                col, row, f"{value:.1%}", ha="center", va="center", fontsize=8,
                color="white" if value > 0.6 * vmax else "#17253d",
            )
    ax.set_xticks(range(len(common)), genres.loc[common, "title"], rotation=45, ha="right")
    ax.set_yticks(range(len(common)), genres.loc[common, "title"])
    ax.set_title("Co-occurrence of direct genre labels")
    fig.colorbar(heatmap, ax=ax, label="Jaccard overlap", format=PercentFormatter(1))
    plt.show()
else:
    print("Co-occurrence requires at least two observed genres.")

### Track map from acoustic features

Each dot is one selected track. The coordinates come from PCA on FMA's 518 published audio features, which summarise properties such as spectral shape, timbre and pitch. Missing feature values are imputed with training medians. The imputer, standardisation and PCA are fit on the published training split. The fitted projection then places tracks from all splits on the same map. [FMA features](https://arxiv.org/html/1612.01840v3#S2.SS5)

Colour shows a track's original top-level FMA branch when it has exactly one. Tracks with several branches or with no branch tag have their own colours. In the plot cell, set **FOCUS_BRANCH** to a name such as **Jazz** and rerun that cell to highlight all tracks with that branch, including multi-label tracks. The two PCA axes retain part of the feature variation, so use the map to explore broad patterns.

The first figure zooms in on the dense part of the map. **ZOOM_TAIL** trims the view at both ends of each axis and the output reports how many dots are visible. The last figure shows the complete range with the same coordinates. For a tighter view, increase **ZOOM_TAIL** in the plot cell and rerun it.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

features = pd.read_csv(
    metadata / "features.csv", index_col=0, header=[0, 1, 2], dtype=np.float32
)
assert features.index.is_unique
feature_values = features.reindex(ids).to_numpy(copy=True)
del features
feature_values[~np.isfinite(feature_values)] = np.nan
has_features = np.isfinite(feature_values).any(axis=1)
training = tracks["split"].eq("training").to_numpy() & has_features
assert training.sum() >= 2, "PCA requires at least two training tracks with acoustic features."

projection = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    PCA(n_components=2, svd_solver="randomized", random_state=0),
)
projection.fit(feature_values[training])
xy = pd.DataFrame(
    projection.transform(feature_values[has_features]),
    index=ids[has_features],
    columns=["PC1", "PC2"],
)
del feature_values
assert np.isfinite(xy.to_numpy()).all()
variance = projection.named_steps["pca"].explained_variance_ratio_
print(
    f"{len(xy):,} of {len(ids):,} selected tracks plotted from published acoustic features. "
    f"{len(ids) - len(xy):,} have no usable feature row."
)

In [ ]:
from io import BytesIO

from IPython.display import Image as NotebookImage
from matplotlib.lines import Line2D
from PIL import Image as PillowImage

FOCUS_BRANCH = None
ZOOM_TAIL = 0.02

branch_names = roots["title"].sort_values()
single = root_tags[root_tags["track_id"].map(roots_per_track).eq(1)]
track_colour = single.set_index("track_id")["genre_id"].map(branch_names)
track_colour = track_colour.reindex(xy.index).fillna("No branch tag")
track_colour.loc[roots_per_track.reindex(xy.index).gt(1)] = "Multiple branches"

palette = [colour for i, colour in enumerate(plt.get_cmap("tab20").colors) if i not in (14, 15)]
branch_colours = dict(zip(branch_names, palette[: len(branch_names)], strict=True))
colours = {"Multiple branches": "#64748b", "No branch tag": "#bcc5d1", **branch_colours}
if FOCUS_BRANCH is not None:
    assert FOCUS_BRANCH in branch_colours, "Choose an original FMA branch from the legend."
    genre_id = branch_names[branch_names.eq(FOCUS_BRANCH)].index[0]
    focus_ids = root_tags.loc[root_tags["genre_id"].eq(genre_id), "track_id"]
    track_colour = pd.Series(
        np.where(xy.index.isin(focus_ids), FOCUS_BRANCH, "Other tracks"), index=xy.index
    )
    colours = {"Other tracks": "#bcc5d1", FOCUS_BRANCH: branch_colours[FOCUS_BRANCH]}


assert 0 <= ZOOM_TAIL < 0.5
bounds = xy.quantile([ZOOM_TAIL, 1 - ZOOM_TAIL])
in_zoom = (xy.ge(bounds.iloc[0]) & xy.le(bounds.iloc[1])).all(axis=1)
muted_groups = {"Multiple branches", "No branch tag", "Other tracks"}
foreground = xy.loc[~track_colour.isin(muted_groups)].sample(frac=1, random_state=0)
foreground_colours = track_colour.loc[foreground.index].map(colours).tolist()
legend = [
    Line2D([], [], marker="o", linestyle="none", color=colour, markersize=6, label=group)
    for group, colour in colours.items()
    if track_colour.eq(group).any()
]
pc_labels = [f"PC{i} ({share:.1%} of training variance)" for i, share in enumerate(variance, 1)]


def show_track_figure(fig):
    with BytesIO() as rendered, BytesIO() as compact:
        fig.savefig(rendered, format="png", bbox_inches="tight")
        with PillowImage.open(rendered) as original:
            palette = original.convert("RGB").quantize(
                colors=96, dither=PillowImage.Dither.NONE
            )
            palette.save(compact, format="PNG", optimize=True)
        display(NotebookImage(data=compact.getvalue(), format="png"), include=["image/png"])
    plt.close(fig)


def plot_track_map(zoom):
    fig, ax = plt.subplots(figsize=(11.6, 7) if zoom else (9.4, 5.8), layout="constrained")
    for group in colours.keys() & muted_groups:
        points = xy.loc[track_colour.eq(group)]
        ax.scatter(
            points["PC1"], points["PC2"], s=3, color=colours[group],
            alpha=0.07, linewidths=0,
        )
    ax.scatter(
        foreground["PC1"], foreground["PC2"], s=4 if zoom else 3,
        color=foreground_colours, alpha=0.30, linewidths=0,
    )
    extent = "central view" if zoom else "full range"
    ax.set(
        xlabel=pc_labels[0], ylabel=pc_labels[1], title=f"FMA track map ({extent})",
    )
    if zoom:
        ax.set(xlim=bounds["PC1"], ylim=bounds["PC2"])
    ax.grid(alpha=0.6)
    ax.legend(
        handles=legend, loc="upper center", bbox_to_anchor=(0.5, -0.12),
        ncols=4, frameon=False,
    )
    show_track_figure(fig)


print(f"Central view: {in_zoom.sum():,} of {len(xy):,} tracks ({in_zoom.mean():.1%}).")
plot_track_map(zoom=True)

#### Compare branches at the same scale

Each panel uses the central view's axes and shows all tracks carrying that branch. Tracks with several branches appear in each applicable panel. The count is the number of points inside the zoom window. Overlapping dots appear darker, which makes the dense regions easier to see. The final plot restores the full range, including the outliers.

In [ ]:
branch_xy = root_tags.join(xy.loc[in_zoom], on="track_id", how="inner")
fig, axes = plt.subplots(
    4, 4, sharex=True, sharey=True, figsize=(12, 9), layout="constrained"
)
for ax, (genre_id, name) in zip(axes.flat, branch_names.items(), strict=True):
    points = branch_xy[branch_xy["genre_id"].eq(genre_id)]
    ax.scatter(
        points["PC1"], points["PC2"], color=branch_colours[name],
        s=3, alpha=0.34, linewidths=0,
    )
    ax.set_title(f"{name} ({len(points):,})", fontsize=10)
    ax.set(xlim=bounds["PC1"], ylim=bounds["PC2"])
    ax.grid(alpha=0.6)
fig.suptitle("Track distributions by FMA branch")
fig.supxlabel(pc_labels[0])
fig.supylabel(pc_labels[1])
show_track_figure(fig)

In [ ]:
plot_track_map(zoom=False)

### Artist concentration

Artists are ranked from most to least tracks. At rank 10, the curve gives the share of the entire selected catalogue belonging to the ten largest artists. A steep rise signals concentration and can make an artist-heavy label look well supported even when it has few independent performers. The rank axis is logarithmic. Tracks with unknown artist IDs contribute to the denominator only.

In [ ]:
def plot_concentration(field, title, colour):
    counts = tracks[field].value_counts()
    ranks = np.arange(1, len(counts) + 1)
    shares = counts.cumsum().to_numpy() / len(tracks)
    fig, ax = plt.subplots(figsize=(8, 3.6), layout="constrained")
    ax.plot(ranks, shares, color=colour, linewidth=2.4)
    ax.fill_between(ranks, shares, color=colour, alpha=0.14)
    ax.set(
        xscale="log", xlabel="Rank by track count", ylim=(0, 1.02),
        ylabel="Cumulative catalogue share", title=title,
    )
    ax.yaxis.set_major_formatter(PercentFormatter(1))
    ax.grid(axis="y")
    plt.show()


plot_concentration("artist_id", "Artist concentration", "#5352a3")

### Published durations

The histogram uses FMA's published duration in seconds. The horizontal axis is logarithmic so short and long recordings are readable on the same chart. The dashed line marks the median of positive, finite values. Section 2 compares these values with the decoded audio.

In [ ]:
durations = tracks["duration"].dropna()
durations = durations[durations.gt(0) & np.isfinite(durations)]

if durations.empty:
    print("No positive, finite durations are available.")
else:
    fig, ax = plt.subplots(figsize=(8, 3.7), layout="constrained")
    bins = np.geomspace(durations.min() * 0.99, durations.max() * 1.01, 36)
    ax.hist(durations, bins=bins, color="#4086bb", edgecolor="white", linewidth=0.7)
    ax.axvline(
        durations.median(), color="#b55426", linestyle="--", linewidth=2,
        label=f"Median {durations.median():g} s",
    )
    ax.set(
        xscale="log", xlabel="Published duration (seconds)", ylabel="Tracks",
        title="Track durations",
    )
    ax.yaxis.set_major_formatter("{x:,.0f}")
    ax.grid(axis="y")
    ax.legend(facecolor="white")
    plt.show()

### Annotation and metadata coverage

Bars show the percentage of **all selected tracks**, with counts printed at the end. The first three bars describe how many direct tags a track has. The fourth counts tracks spanning more than one top-level branch. Missing fields can overlap with each other and with the annotation categories. Treat absent genre tags as unknown.

In [ ]:
labels_per_track = direct.groupby("track_id").size().reindex(ids, fill_value=0)
coverage = pd.Series(
    {
        "No genre tags": labels_per_track.eq(0).sum(),
        "One direct tag": labels_per_track.eq(1).sum(),
        "Multiple direct tags": labels_per_track.gt(1).sum(),
        "Multiple top-level genres": roots_per_track.gt(1).sum(),
        "Missing artist ID": tracks["artist_id"].isna().sum(),
        "Missing album ID": tracks["album_id"].isna().sum(),
        "Missing published split": tracks["split"].isna().sum(),
        "Missing or invalid duration": len(tracks) - len(durations),
    }
)

shares = coverage / len(tracks)
fig, ax = plt.subplots(figsize=(8.6, 4.2), layout="constrained")
bars = ax.barh(
    coverage.index, shares, color=["#5352a3"] * 3 + ["#168477"] + ["#bd6447"] * 4
)
ax.bar_label(
    bars,
    labels=[f"{count:,} ({share:.1%})" for count, share in zip(coverage, shares, strict=True)],
    padding=5,
)
ax.set(xlabel="Share of selected tracks", xlim=(0, 1.18), title="Annotation and metadata coverage")
ax.set_xticks(np.linspace(0, 1, 6))
ax.xaxis.set_major_formatter(PercentFormatter(1))
ax.invert_yaxis()
ax.grid(axis="x")
plt.show()

## 2. Audio audit

A verified checksum shows that a file was downloaded intact, not that it is usable. `atlas fma-audit` decodes every present track from its tar shard and then reads it through the shared loader, so a valid track is one the experiments can load. It records the decoded duration, the peak and RMS level of the whole track and a per-second loudness contour.

- **Excluded** tracks are technical failures: the file cannot be decoded or read by the loader, holds too little audio, or is digitally silent.
- **Warnings** mark unusual but valid audio: a decoded duration that differs from FMA's, a track shorter than one analysis window, or a very quiet recording. These tracks stay in the dataset.
- **Confirmed duplicates** are byte-identical files. **Re-encoding candidates** have almost the same decoded duration and loudness contour but different bytes; they are reported and left unresolved.

The first table lists the thresholds from `music_atlas.audit`.

In [ ]:
audit = pd.read_parquet(data / "manifests/fma_audio_audit.parquet").set_index("track_id")
measured = pd.concat(
    pd.read_parquet(path, columns=["track_id", "peak", "rms_dbfs", "error"])
    for path in sorted((data / "manifests/fma_audio_audit").glob("*.parquet"))
).set_index("track_id")
flags = audit["warning"].str.get_dummies(";").astype(bool)
flags = flags.reindex(columns=["duration_mismatch", "short", "quiet"], fill_value=False)
checks = audit.join(measured[["peak", "rms_dbfs"]]).join(
    tracks[["artist", "title", "artist_id", "duration"]].rename(
        columns={"duration": "published_seconds"}
    )
)
valid = audit["valid"]

display(pd.Series({
    "Excluded: shorter than (s)": rules.MIN_SECONDS,
    "Excluded: no sample above (peak, linear)": rules.SILENT_PEAK,
    "Warning: decoded and published duration differ by more than (s)": rules.DURATION_TOLERANCE,
    "Warning: shorter than one window (s)": rules.SHORT_SECONDS,
    "Warning: RMS below (dBFS)": rules.QUIET_DBFS,
    "Candidate: durations within (s)": rules.MATCH_SECONDS,
    "Candidate: contour correlation at least": rules.MATCH_CORRELATION,
}, name="Threshold").to_frame())
display(pd.Series({
    "Catalogue tracks": len(audit),
    "Present in the inventory": len(ids),
    "Valid": valid.sum(),
    "Excluded": (~valid).sum(),
}, name="Tracks").to_frame())
display(audit["error"].value_counts().rename("Tracks").rename_axis("Exclusion reason").to_frame())
display(flags.sum().rename("Tracks").rename_axis("Warning (track stays valid)").to_frame())
print("Most frequent decoder messages")
display(
    measured["error"].dropna().str.replace(r"<[^>]*>", "<file>", regex=True)
    .value_counts().head(5).rename("Tracks").to_frame()
)

### Where the thresholds sit

Published durations are whole seconds, so decoded and published values normally agree within a second; the tail on the left consists of files that decode to less audio than FMA lists. The level histogram shows how far the quiet warning lies from typical recordings. Both vertical axes are logarithmic.

In [ ]:
difference = (checks["duration_seconds"] - checks["published_seconds"])[valid].dropna()
level = checks.loc[valid, "rms_dbfs"].dropna()
fig, (left, right) = plt.subplots(1, 2, figsize=(11, 3.6), layout="constrained")
left.hist(difference.clip(-10, 10), bins=80, color="#4086bb")
tolerance = rules.DURATION_TOLERANCE
for edge in (-tolerance, tolerance):
    left.axvline(edge, color="#b55426", linestyle="--", linewidth=1.5)
left.set(
    yscale="log", xlabel="Decoded minus published duration (s, clipped to ±10)",
    ylabel="Valid tracks",
    title=f"{flags['duration_mismatch'].sum():,} tracks differ by over {tolerance:g} s",
)
right.hist(level.clip(lower=-70), bins=80, color="#2a9d8f")
right.axvline(rules.QUIET_DBFS, color="#b55426", linestyle="--", linewidth=1.5)
right.set(
    yscale="log", xlabel="RMS level of the whole track (dBFS, clipped at −70)",
    title=f"{flags['quiet'].sum():,} tracks below {rules.QUIET_DBFS:g} dBFS",
)
for ax in (left, right):
    ax.grid(axis="y")
plt.show()

### Problematic cases

Up to three tracks per problem, most severe first. Durations are in seconds and **peak** is linear, where 1.0 is full scale.

In [ ]:
columns = ["artist", "title", "published_seconds", "duration_seconds", "peak", "rms_dbfs"]
listed = checks["published_seconds"]
ratio = checks["duration_seconds"] / listed.where(listed > 0)
cases = {
    "Cannot be decoded or read by the loader": checks[
        checks["error"].isin(["decode_error", "loader_error"])
    ],
    "Digitally silent": checks[checks["error"].eq("silent")],
    f"Under {rules.MIN_SECONDS:g} s of audio": checks[checks["error"].eq("too_short")],
    "Decodes to much less than the published duration": checks[flags["duration_mismatch"]]
    .assign(ratio=ratio).sort_values("ratio"),
    "Quietest valid tracks": checks[flags["quiet"]].sort_values("rms_dbfs"),
    "Shortest valid tracks": checks[flags["short"]].sort_values("duration_seconds"),
}
for title, rows in cases.items():
    print(f"{title}: {len(rows):,} tracks")
    if len(rows):
        display(rows[columns].head(3).round(2))

### Duplicates and re-encoding candidates

Artist grouping already keeps an artist's repeated uploads in one partition. Groups that span **several artists** are the ones a split by artist alone would separate, and groups across the **published splits** are leakage in FMA's own partition. The second table shows example candidates.

In [ ]:
grouped = audit.join(tracks[["artist_id", "split"]])


def group_report(column):
    groups = grouped.dropna(subset=[column]).groupby(column)
    return {
        "Groups": groups.ngroups,
        "Tracks": int(groups.size().sum()),
        "Groups with several artists": int(groups["artist_id"].nunique().gt(1).sum()),
        "Groups across published splits": int(groups["split"].nunique().gt(1).sum()),
    }


display(pd.DataFrame({
    "Confirmed duplicates (same SHA-1)": group_report("duplicate_group"),
    "Re-encoding candidates": group_report("reencoding_candidate"),
}).T)
candidates = checks.dropna(subset=["reencoding_candidate"]).sort_values(
    ["reencoding_candidate", "duration_seconds"]
)
columns = ["reencoding_candidate", "artist", "title", "duration_seconds"]
display(candidates[columns].head(8).round(2))

### Listen to a track

Add track IDs from the tables above to **LISTEN** to hear the centred window the quick profile uses, read from the shard through the shared loader. The list is empty in the saved notebook so that it stores no audio.

In [ ]:
LISTEN = []

settings = profile("quick")
for track_id in LISTEN:
    row = inventory.loc[track_id]
    with tarfile.open(data / row["shard"]) as tar:
        mp3 = tar.extractfile(row["audio_path"]).read()
    [window] = load_windows(
        io.BytesIO(mp3), track_id, sample_rate=settings["sample_rate"],
        window_seconds=settings["window_seconds"], windows_per_track=1,
    )
    print(f"{track_id}: {checks.at[track_id, 'artist']} – {checks.at[track_id, 'title']}")
    display(Audio(window.samples, rate=window.sample_rate))

## 3. Retained, excluded and unresolved tracks

Every catalogue row stays in the manifest. A valid track is used; an excluded one keeps its reason and receives no partition; a warning or an unresolved candidate changes nothing by itself and is counted here so that its effect can be judged.

In [ ]:
untagged = pd.Series(~audit.index.isin(root_tags["track_id"]), index=audit.index)
outcome = pd.Series({
    "Valid": valid.sum(),
    **{f"Excluded: {reason}": count for reason, count in audit["error"].value_counts().items()},
    "Valid with at least one warning": (valid & flags.any(axis=1)).sum(),
    "Valid, in a confirmed duplicate group": (valid & audit["duplicate_group"].notna()).sum(),
    "Valid, unresolved re-encoding candidate": (
        valid & audit["reencoding_candidate"].notna()
    ).sum(),
    "Valid without a top-level genre": (valid & untagged).sum(),
    "Eligible as a listening example": audit["listening_example"].sum(),
}, name="Tracks")
display(outcome.to_frame())

### Effect on each genre

Exclusions are counted against all present tracks of a branch; a track with several branches counts in each. The warning columns and the share of tracks **over 10 minutes** use valid tracks and decoded durations. They show what a duration or loudness cutoff would remove from each branch, which is why none is applied: the loader pads a short track and samples a fixed number of windows from a long one.

In [ ]:
tagged = (
    root_tags.join(valid, on="track_id").join(flags, on="track_id")
    .join(checks[["artist_id", "duration_seconds"]], on="track_id")
)
tagged["genre"] = tagged["genre_id"].map(genres["title"])
tagged["over_10_min"] = tagged["duration_seconds"].gt(600)
every = tagged.groupby("genre")
kept = tagged[tagged["valid"]].groupby("genre")
effect = pd.DataFrame({
    "Tracks": every.size(),
    "Excluded": every.size() - kept.size(),
    "Excluded (%)": 100 * (1 - kept.size() / every.size()),
    "Artists": every["artist_id"].nunique(),
    "Artists lost": every["artist_id"].nunique() - kept["artist_id"].nunique(),
    "Duration mismatch (%)": 100 * kept["duration_mismatch"].mean(),
    "Under 10 s (%)": 100 * kept["short"].mean(),
    "Quiet (%)": 100 * kept["quiet"].mean(),
    "Over 10 min (%)": 100 * kept["over_10_min"].mean(),
}).sort_values("Tracks", ascending=False)
display(effect.round(2))

### Rules applied

- **Excluded tracks** are technical failures only. They remain in `tracks.parquet` with their reason.
- **Warnings do not exclude.** Short pieces and quiet recordings are valid music, and a truncated file still carries its genre. No duration cutoff is applied.
- **Confirmed duplicates** join their artists into one group, so all copies share a partition.
- **Re-encoding candidates stay unresolved** and outside the duplicate groups. Candidates by one artist already share a partition; candidates across artists are the ones to confirm by listening.
- **Missing genre tags are unknown, not negative.** Untagged valid tracks keep a partition for retrieval and are left out of supervised genre scoring.
- **Listening examples** come from valid tracks under CC0, Public Domain Mark, CC BY or CC BY-SA, which allow sharing adapted excerpts; credit the artist, title and licence.

## 4. Grouping and splitting

`atlas fma-split --audit` assigns whole groups to train, validation and test in an 80/10/10 ratio. A group is a connected component of valid tracks linked by a shared artist or a confirmed duplicate, so neither can appear in two partitions. Tracks without a known artist receive no partition.

The tables compare three assignments of the same eligible tracks: the **published** FMA split, a seeded random assignment of groups (**group_random**) and the **canonical** split, which starts from the random one and moves groups until each top-level genre has its catalogue share of tracks and of independent groups in every partition. The first table identifies the dataset and its inputs, the second counts tracks per partition, and the third counts artists, duplicate groups and components found in more than one partition.

In [ ]:
ready = data / "ready"
result = ready / VERSION
if not (result / "tracks.parquet").is_file():
    available = sorted(p.parent.name for p in ready.glob("*/tracks.parquet"))
    raise ValueError(f"Set VERSION to one of: {', '.join(available)}")
summary = json.loads((result / "summary.json").read_text(encoding="utf-8"))
support = pd.read_csv(result / "class_support.csv")
versions = [VERSION]
if COMPARE_VERSION and COMPARE_VERSION != VERSION:
    comparison_dir = ready / COMPARE_VERSION
    if (comparison_dir / "tracks.parquet").is_file():
        other_summary = json.loads((comparison_dir / "summary.json").read_text(encoding="utf-8"))
        if all(summary[key] == other_summary[key] for key in ("inputs", "schema", "eligible")):
            versions.append(COMPARE_VERSION)
        else:
            print(f"{COMPARE_VERSION}: choose a comparison version with the same inputs.")
    else:
        print(f"{COMPARE_VERSION}: comparison manifest unavailable.")
status = "Audited dataset" if summary.get("audio_audited") else "Pre-audit dataset"
print(f"{VERSION} | {status} | {summary['version']}")
display(pd.Series(
    {**summary["inputs"], "tracks.parquet": summary["manifest_sha256"]}, name="SHA-256"
).to_frame())
display(pd.DataFrame(summary["splits"]).T)
display(pd.DataFrame(summary["overlap"]).T)

### Label support and frequency baselines

The first table shows the largest shift in label prevalence in percentage points, relative to the selected annotated catalogue. The next tables count positive tracks and independent groups per class. The all-negative baseline can achieve high label accuracy by predicting the common absence of tags. Macro recall and macro-F1 give equal weight to each scored class and expose that failure.

In [ ]:
display(support.groupby(["scheme", "split"])["prevalence_shift"].apply(
    lambda values: 100 * values.abs().max()
).round(2).rename("Max absolute shift (pp)").unstack())
canonical = support[support["scheme"].eq("canonical")]
display(canonical.pivot(index="genre", columns="split", values=["tracks", "groups"]))
for split, scores in summary["baselines"].items():
    print(f"{split}: constant-score mAP {scores['constant_score_map']:.3f}")
    display(pd.DataFrame({k: v for k, v in scores.items() if isinstance(v, dict)}).T.round(3))

### Proportional and rare-support splits

The rare-support strategy raises sparse genres to as many as 250 positives in each held-out partition, capped at 20% of the genre's tracks. It moves tracks out of training, so the two versions are separate datasets. Compare held-out positive tracks and independent groups per genre. Both versions partition the same catalogue. Use the three files from one version together.

In [ ]:
if len(versions) == 2:
    other_version = versions[1]
    other = pd.read_csv(ready / other_version / "class_support.csv")
    comparison = pd.concat([
        canonical.assign(version=VERSION),
        other[other["scheme"].eq("canonical")].assign(version=other_version),
    ])
    comparison = comparison[comparison["split"].isin(["validation", "test"])].copy()
    comparison["Tracks / groups"] = (
        comparison["tracks"].astype(str) + " / " + comparison["groups"].astype(str)
    )
    display(comparison.pivot(index="genre", columns=["version", "split"],
                             values="Tracks / groups"))

## 5. The published FMA split and ours

The first table shows what each version contains. The second checks that the manifest's expanded labels equal FMA's published `genres_all`. The third counts artists, albums, confirmed duplicate groups and re-encoding candidate groups found in more than one partition, for the same tracks under the published assignment and under ours. Our allocator keeps artists and confirmed duplicates together; a release with several artists can still span partitions, so albums measure a separate source of dependence. Candidates stay unresolved and are not used for grouping (section 3), so their column is the leakage this choice leaves.

In [ ]:
manifests = {}
contents = {}
balanced = {}
for version in versions:
    directory = ready / version
    columns = [
        "track_id", "split", "artist_id", "album_id",
        "duration_seconds", "genre_ids", "duplicate_group",
    ]
    manifest = pd.read_parquet(directory / "tracks.parquet", columns=columns)
    manifests[version] = manifest.loc[manifest["split"].notna()].set_index("track_id")
    contents[version] = {
        "Catalogue rows": len(manifest),
        "Assigned tracks": len(manifests[version]),
        "Ineligible rows": manifest["split"].isna().sum(),
        "Missing artist": manifest["artist_id"].isna().sum(),
        "Missing album": manifest["album_id"].isna().sum(),
        "Missing duration": manifest["duration_seconds"].isna().sum(),
        "No genre annotation": manifest["genre_ids"].map(len).eq(0).sum(),
    }
    if (directory / "balanced_eval.parquet").is_file():
        view = pd.read_parquet(directory / "balanced_eval.parquet")
        for split, rows in view.groupby("split"):
            counts = rows.groupby(["genre_id", "label_present"]).size()
            assert counts.nunique() == 1
            balanced[version, split] = {
                "Scored genres": rows["genre_id"].nunique(),
                "Tracks per genre per side": counts.iloc[0],
                "Track–genre rows": len(rows),
            }
    else:
        print(f"{version}: balanced evaluation file unavailable.")
display(pd.DataFrame(contents).T)

published = pd.DataFrame({
    "published_genres": labels["genres_all"].map(frozenset),
    "published_split": tracks["split"].replace("training", "train"),
})
root_ids = frozenset(roots.index)
agreement = {}
leakage = {}
branch_profiles = {}
for version, assigned in manifests.items():
    rows = assigned.join(published, validate="one_to_one").join(audit["reencoding_candidate"])
    reference_tags = rows["published_genres"].map(
        lambda tags: tags if isinstance(tags, frozenset) else frozenset()
    )
    manifest_tags = rows["genre_ids"].map(frozenset)
    agreement[version] = {
        "Tracks compared": len(rows),
        "Missing FMA metadata rows": len(rows.index.difference(published.index)),
        "Different expanded tags": manifest_tags.ne(reference_tags).sum(),
        "Different top-level tags": (
            manifest_tags.map(root_ids.intersection)
            .ne(reference_tags.map(root_ids.intersection)).sum()
        ),
    }
    assignments = {"Published FMA": "published_split", "Versioned": "split"}
    for assignment, split_column in assignments.items():
        leakage[version, assignment] = {
            label: rows.groupby(field)[split_column].nunique().gt(1).sum()
            for field, label in {
                "artist_id": "Artists", "album_id": "Albums",
                "duplicate_group": "Duplicate groups",
                "reencoding_candidate": "Re-encoding candidate groups",
            }.items()
        }
    branches = rows.assign(genre_id=reference_tags).explode("genre_id")
    branch_profiles[version] = branches.loc[
        branches["genre_id"].isin(roots.index), ["split", "genre_id", "artist_id"]
    ]

display(pd.DataFrame(agreement).T)
display(pd.DataFrame(leakage).T.rename_axis(["Version", "Assignment"]))

### Artist concentration within each genre

Disjoint artists do not guarantee a diverse evaluation. If one artist supplies most of a genre's held-out tracks, the score for that genre mostly measures that artist, and extra tracks add support without adding independent examples. For every top-level genre the table gives the tracks, the distinct artists and the share of the largest artist, in the whole assigned catalogue and in each held-out partition. Rows are ordered by the largest held-out share, and the list below the table names every genre and partition where one artist supplies at least a third of the tracks.

In [ ]:
DOMINANT_SHARE = 1 / 3


def concentration(rows):
    keys = ["split", "genre_id"]
    stats = rows.groupby(keys).agg(Tracks=("artist_id", "size"), Artists=("artist_id", "nunique"))
    largest = rows.groupby([*keys, "artist_id"]).size().groupby(level=[0, 1]).max()
    stats["Top artist (%)"] = 100 * largest / stats["Tracks"]
    return stats


for version, branches in branch_profiles.items():
    stats = concentration(pd.concat([branches.assign(split="catalogue"), branches]))
    table = stats.unstack("split").swaplevel(axis=1)
    table = table[[
        (split, column) for split in ("catalogue", "validation", "test")
        for column in ("Tracks", "Artists", "Top artist (%)")
    ]]
    held_out = table.xs("Top artist (%)", axis=1, level=1)[["validation", "test"]]
    table = table.loc[held_out.max(axis=1).sort_values(ascending=False).index]
    table.index = pd.Index(table.index.map(roots["title"]), name="Genre")
    print(version)
    display(table.round(1))
    dominated = held_out.stack()
    dominated = dominated[dominated.ge(100 * DOMINANT_SHARE)]
    print("One artist supplies at least a third of the held-out tracks:", ", ".join(
        f"{roots.at[genre, 'title']} {split} ({share:.0f}%)"
        for (genre, split), share in dominated.items()
    ) or "no genre")

## 6. Primary benchmark, supplementary views and limitations

**Primary benchmark.** `VERSION`, the proportional split of the audited catalogue, is shared by every representation. Validation and test keep the natural prevalence of each genre. Report macro-F1, per-genre recall and mAP alongside micro-F1, next to the frequency baselines of section 4. Select models and thresholds on validation and evaluate once on test.

**Supplementary views.** `COMPARE_VERSION` is a separate train/validation/test partition with more held-out support for sparse genres; compare models within one version only. `balanced_eval.parquet` samples, for each scored genre, equally many tagged and untagged-by-class tracks from the existing validation and test partitions, preferring distinct groups. It is a diagnostic for per-class errors at equal support and is summarised below. Keep both apart from the primary test metrics.

**Comparability.** Results on these partitions use our protocol. They are not directly comparable with published FMA benchmarks, which use the published split and mostly the single-label small and medium subsets.

In [ ]:
if balanced:
    display(pd.DataFrame(balanced).T.rename_axis(["Version", "Split"]))

**Remaining limitations**

- Genre tags are incomplete: a missing tag is unknown, and tagged negatives may be wrong.
- Re-encoding candidates are unresolved. Candidates across artists may place the same recording in two partitions; section 5 counts the groups that do.
- Sparse genres stay concentrated on few artists in the held-out partitions (section 5); read their scores with the artist counts.
- Albums can span partitions when a release has several artists.
- Truncated, short and quiet tracks are kept with a warning; their effect per genre is in section 3.
- The audit decodes with libsndfile. A file that only another decoder can read is excluded.